## **Statistical Testing Using k-Fold Cross Validation**

To make sure the difference between the accuracy and precision of Random Forest and the other models is statistically significant, we have to perform paired t-tests using k-fold cross validation.

This splits the test set into k partitions, testing the model's accuracy and precision k different times. The average accuracy and precision of each model will then be compared with the average accuracy and precision of Random Forest through a paired t-test, revealing the significant difference between the performance of the two models.

---

In [6]:
import numpy as np
import pandas as pd

from scipy.stats import ttest_rel

from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

In [7]:
%store -r
print("Variables restored successfully.")

Variables restored successfully.


In [8]:
print(f"X_train_scaled                              : {X_train_scaled.shape}")
print(f"X_val_scaled                                : {X_val_scaled.shape}")
print(f"X_test_scaled                               : {X_test_scaled.shape}")
print(f"y_train                                     : {y_train.shape}")
print(f"y_val                                       : {y_val.shape}")
print(f"y_test                                      : {y_test.shape}")
print(f"X_clean                                     : {X_clean.shape}")

X_train_scaled                              : (29514, 395)
X_val_scaled                                : (9838, 395)
X_test_scaled                               : (9839, 395)
y_train                                     : (29514,)
y_val                                       : (9838,)
y_test                                      : (9839,)
X_clean                                     : (49191, 395)


## **k-Fold Cross Validation**

In [9]:
knn_model = KNeighborsClassifier(n_neighbors=29)

lr_model = LogisticRegression(
    C=0.1,
    penalty='l1',
    solver='liblinear',
    random_state=42,
    **{}
)

svm_model = SVC(
    kernel='rbf',
    C=1,
    random_state=42
)

nn_model = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    alpha=0.001,
    solver='adam',
    random_state=42
)

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    bootstrap=True,
    random_state=42,
    n_jobs=-1
)

gnb_model = GaussianNB()

models = {
    "Logistic Regression": lr_model,
    "SVM": svm_model,
    "Neural Network": nn_model,
    "Gaussian Naive Bayes": gnb_model,
    "Random Forest": rf_model,
    "K-Nearest Neighbors": knn_model
}

folds = [3, 5, 7, 10, 15]
results_summary = []

print(f"{'K-Fold':<10} | {'Model':<25} | {'Mean Acc':<10} | {'Std Dev':<10}")

ttest_results = []

for k in folds:
    kfold = KFold(n_splits=k, shuffle=True, random_state=42)
    
    cv_scores = {name: [] for name in models.keys()}
    
    for name, model in models.items():
        scores = cross_val_score(model, X_train_scaled, y_train, cv=kfold, scoring='accuracy')
        cv_scores[name] = scores
        
        print(f"{k:<10} | {name:<25} | {scores.mean():.4f}   | {scores.std():.4f}")
       
    rf_scores = np.array(cv_scores["Random Forest"])
    
    print(f"\nPaired T-Test Results (K={k}) vs Random Forest Baseline:")
    print(f"{'Model':<25} | {'T-Statistic':<15} | {'P-Value':<15} | {'Significant (α=0.05)':<25}")
    
    for name, scores in cv_scores.items():
        if name == "Random Forest":
            continue
            
        model_scores = np.array(scores)
        
        t_stat, p_val = ttest_rel(model_scores, rf_scores)
        
        significant = "Yes" if p_val < 0.05 else "No"
        diff_direction = "Better" if model_scores.mean() > rf_scores.mean() else "Worse"
        
        print(f"{name:<25} | {t_stat:<15.4f} | {p_val:<15.4f} | {significant} ({diff_direction})")
        
        ttest_results.append({
            "K": k,
            "Model": name,
            "T_Stat": t_stat,
            "P_Value": p_val,
            "Significant": significant,
            "Direction": diff_direction
        })
    
    print("\n")

print("=" * 80)
print("SUMMARY OF STATISTICALLY SIGNIFICANT DIFFERENCES (P < 0.05)")
print("=" * 80)
sig_df = pd.DataFrame(ttest_results)
if not sig_df[sig_df['Significant'] == "Yes"].empty:
    print(sig_df[sig_df['Significant'] == "Yes"][["K", "Model", "P_Value", "Direction"]])
else:
    print("No models showed a statistically significant difference from Random Forest across the tested K-folds.")
print("=" * 80)

K-Fold     | Model                     | Mean Acc   | Std Dev   


c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty=

3          | Logistic Regression       | 0.7279   | 0.0032
3          | SVM                       | 0.7423   | 0.0031
3          | Neural Network            | 0.7081   | 0.0040
3          | Gaussian Naive Bayes      | 0.6594   | 0.0031
3          | Random Forest             | 0.7527   | 0.0023
3          | K-Nearest Neighbors       | 0.7146   | 0.0046

Paired T-Test Results (K=3) vs Random Forest Baseline:
Model                     | T-Statistic     | P-Value         | Significant (α=0.05)     
Logistic Regression       | -13.7631        | 0.0052          | Yes (Worse)
SVM                       | -6.0819         | 0.0260          | Yes (Worse)
Neural Network            | -35.1106        | 0.0008          | Yes (Worse)
Gaussian Naive Bayes      | -127.0756       | 0.0001          | Yes (Worse)
K-Nearest Neighbors       | -21.4421        | 0.0022          | Yes (Worse)




c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty=

5          | Logistic Regression       | 0.7282   | 0.0053
5          | SVM                       | 0.7433   | 0.0020
5          | Neural Network            | 0.7092   | 0.0081
5          | Gaussian Naive Bayes      | 0.6591   | 0.0035
5          | Random Forest             | 0.7504   | 0.0045
5          | K-Nearest Neighbors       | 0.7128   | 0.0077

Paired T-Test Results (K=5) vs Random Forest Baseline:
Model                     | T-Statistic     | P-Value         | Significant (α=0.05)     
Logistic Regression       | -12.0489        | 0.0003          | Yes (Worse)
SVM                       | -5.4918         | 0.0054          | Yes (Worse)
Neural Network            | -15.3130        | 0.0001          | Yes (Worse)
Gaussian Naive Bayes      | -24.6201        | 0.0000          | Yes (Worse)
K-Nearest Neighbors       | -18.3860        | 0.0001          | Yes (Worse)




...

7          | Logistic Regression       | 0.7292   | 0.0070
7          | SVM                       | 0.7436   | 0.0056
7          | Neural Network            | 0.7081   | 0.0078
7          | Gaussian Naive Bayes      | 0.6582   | 0.0070
7          | Random Forest             | 0.7542   | 0.0067
7          | K-Nearest Neighbors       | 0.7130   | 0.0042

Paired T-Test Results (K=7) vs Random Forest Baseline:
Model                     | T-Statistic     | P-Value         | Significant (α=0.05)     
Logistic Regression       | -12.4635        | 0.0000          | Yes (Worse)
SVM                       | -6.0366         | 0.0009          | Yes (Worse)
Neural Network            | -41.4605        | 0.0000          | Yes (Worse)
Gaussian Naive Bayes      | -30.6889        | 0.0000          | Yes (Worse)
K-Nearest Neighbors       | -28.7356        | 0.0000          | Yes (Worse)




c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty=

10         | Logistic Regression       | 0.7290   | 0.0088
10         | SVM                       | 0.7438   | 0.0056
10         | Neural Network            | 0.7069   | 0.0076
10         | Gaussian Naive Bayes      | 0.6591   | 0.0073
10         | Random Forest             | 0.7530   | 0.0072
10         | K-Nearest Neighbors       | 0.7143   | 0.0117

Paired T-Test Results (K=10) vs Random Forest Baseline:
Model                     | T-Statistic     | P-Value         | Significant (α=0.05)     
Logistic Regression       | -14.2590        | 0.0000          | Yes (Worse)
SVM                       | -4.2772         | 0.0021          | Yes (Worse)
Neural Network            | -16.2158        | 0.0000          | Yes (Worse)
Gaussian Naive Bayes      | -26.4809        | 0.0000          | Yes (Worse)
K-Nearest Neighbors       | -15.6247        | 0.0000          | Yes (Worse)




c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\lance\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty=

15         | Logistic Regression       | 0.7301   | 0.0090
15         | SVM                       | 0.7426   | 0.0082
15         | Neural Network            | 0.7130   | 0.0107
15         | Gaussian Naive Bayes      | 0.6603   | 0.0083
15         | Random Forest             | 0.7526   | 0.0077
15         | K-Nearest Neighbors       | 0.7142   | 0.0109

Paired T-Test Results (K=15) vs Random Forest Baseline:
Model                     | T-Statistic     | P-Value         | Significant (α=0.05)     
Logistic Regression       | -11.5773        | 0.0000          | Yes (Worse)
SVM                       | -4.7373         | 0.0003          | Yes (Worse)
Neural Network            | -14.1969        | 0.0000          | Yes (Worse)
Gaussian Naive Bayes      | -34.8327        | 0.0000          | Yes (Worse)
K-Nearest Neighbors       | -15.5651        | 0.0000          | Yes (Worse)


SUMMARY OF STATISTICALLY SIGNIFICANT DIFFERENCES (P < 0.05)
     K                 Model       P_Value Direction
0   